# 01 — Finance inventory (the supply layer)

The **FINANCE / supply** layer of the city-action fundability model. This single notebook (a) **harmonizes** the ten vetted Chile climate-finance source reviews into one fund inventory, (b) **profiles** that supply side, and (c) classifies each city action by a **financing-availability (coverage)** level. It writes two outputs the model consumes:

- `data/chile_finance_inventory.csv` — one row per fund (99), harmonized schema.
- `data/financing_coverage_by_action.csv` — 102 actions × `coverage_level`.

Coverage is a *where-to-look / gap* signal, not fundability (see `methodology.md` §A0). Sibling source CSVs are read at `../../../../cl-*/...` and are authoritative for provenance and licence. Refresh: re-run this notebook after any source release.

*(The reproducible extraction harness — fetch source pages + one reusable prompt — is `extract_inventory.ipynb` in this release; the source reviews point at it to refresh their fund rows.)*

## A. Harmonize ten source reviews → inventory

# Harmonize — Chile climate-finance inventory (OEF)

Harmonize the four Chile climate-finance source reviews into one inventory.
Inputs (authoritative per-source reviews):
  ../../cl-mma/cl-mma-fondos/releases/v1/data/cl_mma_fondos_v1.csv
  ../../cl-minenergia/cl-minenergia-fondos/releases/v1/data/cl_minenergia_programs_v1.csv
  ../../cl-subdere/cl-subdere-fondos/releases/v1/data/cl_subdere_programs_v1.csv
  ../../cl-minvu/cl-minvu-fondos/releases/v1/data/cl_minvu_programs_v1.csv
Output: chile_finance_inventory.csv  (one row per fund/programme line)
This is an OEF exploratory product — NOT a Mage pipeline. Re-run after any source release.

In [1]:
import pandas as pd, json, os
HERE=os.getcwd()  # notebook runs in its own dir
SRC={
 "cl-mma":      "../../../../cl-mma/cl-mma-fondos/releases/v1/data/cl_mma_fondos_v1.csv",
 "cl-minenergia":"../../../../cl-minenergia/cl-minenergia-fondos/releases/v1/data/cl_minenergia_programs_v1.csv",
 "cl-subdere":  "../../../../cl-subdere/cl-subdere-fondos/releases/v1/data/cl_subdere_programs_v1.csv",
 "cl-minvu":    "../../../../cl-minvu/cl-minvu-fondos/releases/v1/data/cl_minvu_programs_v1.csv",
 "cl-gore":     "../../../../cl-gore/cl-gore-fndr/releases/v1/data/cl_gore_fndr_programs_v1.csv",
 "cl-corfo":    "../../../../cl-corfo/cl-corfo-finance/releases/v1/data/cl_corfo_programs_v1.csv",
 "cl-conaf":    "../../../../cl-conaf/cl-conaf-fondos/releases/v1/data/cl_conaf_programs_v1.csv",
 "cl-mtt":      "../../../../cl-mtt/cl-mtt-fondos/releases/v1/data/cl_mtt_programs_v1.csv",
 "cl-indap":    "../../../../cl-indap/cl-indap-fondos/releases/v1/data/cl_indap_fondos_v1.csv",
 "cl-mop":      "../../../../cl-mop/cl-mop-fondos/releases/v1/data/cl_mop_fondos_v1.csv",
}
SCHEMA=["source_dataset","funder_institution","program_name","program_family","eligible_actor",
 "instrument_type","amount_clp","amount_note","open_date","close_date","status","recurrence",
 "specificity","climate_relevance","climate_relevance_norm","gpc_sectors","access_pathway",
 "detail_level","status_as_of","source_url","notes"]

def norm_relevance(v):
    s=str(v).lower()
    if s.startswith("explicit-adjacent"): return "adjacent"
    if s.startswith("explicit"): return "explicit"
    if "adjacent" in s: return "adjacent"
    if s.startswith("indirect") or "indirect" in s: return "indirect"
    return "unknown"

def load(ds, path):
    df=pd.read_csv(os.path.join(HERE,path))
    out=pd.DataFrame(index=df.index)
    out["source_dataset"]=ds
    if ds=="cl-mma":
        out["funder_institution"]="Ministerio del Medio Ambiente (MMA)"
        out["program_name"]=df["fund_name"]
        out["program_family"]=df["stream"]
        out["amount_clp"]=df["amount_clp"]
        out["amount_note"]=df["amount_suspect"].map(lambda x:"amount_suspect (source typo)" if x else "")
        out["open_date"]=df["open_date"]; out["close_date"]=df["close_date"]
        out["notes"]=df["fund_program"].astype(str)
    else:
        out["funder_institution"]=df["funder_institution"]
        out["program_name"]=df["program"]
        out["program_family"]=df["program"]
        out["amount_clp"]=pd.NA
        out["amount_note"]=df["amount_note"]
        out["open_date"]=pd.NA; out["close_date"]=pd.NA
        out["notes"]=df["notes"]
    for c in ["eligible_actor","instrument_type","status","recurrence","specificity",
              "climate_relevance","gpc_sectors","access_pathway","detail_level","status_as_of","source_url"]:
        out[c]=df[c]
    out["climate_relevance_norm"]=out["climate_relevance"].map(norm_relevance)
    return out[SCHEMA]

frames=[load(ds,p) for ds,p in SRC.items()]
inv=pd.concat(frames,ignore_index=True)

/sessions/focused-confident-knuth/tmp/ipykernel_12/3728976576.py:55: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  inv=pd.concat(frames,ignore_index=True)


## Validate & export

In [2]:
# validate
counts=inv.source_dataset.value_counts().to_dict()
assert counts=={"cl-mma":55,"cl-minenergia":6,"cl-subdere":4,"cl-minvu":4,"cl-gore":4,"cl-corfo":5,"cl-conaf":2,"cl-mtt":3,"cl-indap":9,"cl-mop":7}, counts
assert len(inv)==99, len(inv)
assert inv.source_url.str.startswith("http").all()
assert set(inv.climate_relevance_norm)<= {"explicit","adjacent","indirect","unknown"}
assert inv.program_name.notna().all()
inv.to_csv(os.path.join(HERE,"data/chile_finance_inventory.csv"),index=False)

## B. Profile the supply side

# Chile climate-finance inventory — exploration

Supply-side profiling of the harmonized inventory (`chile_finance_inventory.csv`, built by `harmonize.py`).
Goal: understand what funding exists *before* matching it to actions — by sector, eligible actor, recurrence, specificity, and current usability. Methodology: `methodology.md`.

In [3]:
import pandas as pd, json
inv=pd.read_csv("data/chile_finance_inventory.csv")
print(inv.shape, "rows x cols")
inv.source_dataset.value_counts()

(99, 21) rows x cols


source_dataset
cl-mma           55
cl-indap          9
cl-mop            7
cl-minenergia     6
cl-corfo          5
cl-subdere        4
cl-minvu          4
cl-gore           4
cl-mtt            3
cl-conaf          2
Name: count, dtype: int64

## Usability flag (methodology §2a): open/rolling, or reliably annual/ongoing.

In [4]:
def usable(r):
    s=str(r["status"]).lower(); rec=str(r["recurrence"]).lower()
    if "open" in s or "rolling" in s: return True
    if rec.startswith("annual") or rec.startswith("ongoing"): return True
    return False
inv["usable_now"]=inv.apply(usable, axis=1)
inv.groupby(["source_dataset","usable_now"]).size().unstack(fill_value=0)

usable_now,False,True
source_dataset,,
cl-conaf,0,2
cl-corfo,0,5
cl-gore,0,4
cl-indap,1,8
cl-minenergia,2,4
cl-minvu,0,4
cl-mma,27,28
cl-mop,3,4
cl-mtt,1,2


## Coverage by GPC sector (gpc_sectors is a JSON list — explode it).

In [5]:
ex=inv.assign(gpc=inv.gpc_sectors.apply(json.loads)).explode("gpc")
ex.groupby("gpc").agg(funds=("program_name","count"),
                      usable=("usable_now","sum")).sort_values("funds",ascending=False)

,funds,usable
gpc,,
cross_sector,47,30
afolu,30,17
waste,19,17
stationary_energy,17,12
water,8,7
transportation,6,2
industry,4,4
buildings,2,1


## Eligible actor — who can actually pursue these.

In [6]:
inv["actor_simple"]=inv.eligible_actor.str.extract(r'^(municipality|community/citizen org|household|indigenous community|research/university|school sostenedor|municipality \+ |municipality \()', expand=False).fillna(inv.eligible_actor.str.slice(0,24))
inv.eligible_actor.value_counts()

eligible_actor
unspecified                                                                                                                                29
municipality                                                                                                                               14
community/citizen org                                                                                                                       7
indigenous community                                                                                                                        7
research/university + ngo                                                                                                                   2
usuario/a INDAP (AFC) con orientacion comercial                                                                                             1
public transport bus operator (private firm/person)                                                                                  

## Recurrence & specificity — the scoring controls.

In [7]:
print("recurrence:\n", inv.recurrence.value_counts().to_string())
print("\nspecificity (broad funds are capped at 'Moderate' in scoring):")
print(inv.groupby("specificity").program_name.count().to_string())
print("\nbroad funds:", inv[inv.specificity=='broad'].program_name.tolist())

recurrence:
 recurrence
annual                                                            37
sporadic                                                          16
one-off                                                           11
ongoing                                                            9
ongoing (programme)                                                3
ongoing/periodic                                                   3
time-boxed                                                         3
ongoing (rolling, per-comuna enrolment)                            1
biennial-cycle                                                     1
annual (reajuste en marzo)                                         1
periodic                                                           1
ongoing (per-region calls; reform from 2024)                       1
sporadic (programme rollout)                                       1
ongoing (rolling, per-comuna PDA)                                  1
ongoing (a

## Where could 'Strong' matches come from? sector-specific + usable_now + explicit climate relevance.

In [8]:
strong_supply=inv[(inv.specificity=="sector-specific") & (inv.usable_now) & (inv.climate_relevance_norm=="explicit")]
print(len(strong_supply),"funds qualify as Strong-eligible supply")
strong_supply[["source_dataset","program_name","eligible_actor","recurrence"]].head(20)

47 funds qualify as Strong-eligible supply


,source_dataset,program_name,eligible_actor,recurrence
0,cl-mma,FPA 2026 - Proyectos Sustentables Ciudadanos,community/citizen org,annual
1,cl-mma,FPA 2026 - Proyectos Sustentables en Estableci...,community/citizen org,annual
2,cl-mma,FPA 2026 - Proyectos Sustentables para Pueblos...,indigenous community,annual
6,cl-mma,FPR 2026 - Fondo para el Reciclaje,municipality,annual
16,cl-mma,FPA 2024 – Fortalecimiento para recicladores d...,unspecified,annual
18,cl-mma,FPA 2024 – Chiloé Reduce en tu Establecimiento...,unspecified,annual
21,cl-mma,FPA 2023 – Proyectos Sustentables para Pueblos...,indigenous community,annual
22,cl-mma,FPA 2023 – Proyectos Sustentables en Estableci...,unspecified,annual
23,cl-mma,FPA 2023 – Proyectos Sustentables Ciudadanos,unspecified,annual
24,cl-mma,FPA 2022 – Emprendimientos Verdes para Comunid...,indigenous community,annual


## Read-out (exploratory)

- The inventory is **mitigation-heavy and currently mostly closed** (annual environment funds between cycles), so usability hinges on the `recurrence=annual` rule, not live `status`.
- **Broad funds are few but powerful** (PMU/PMB/FRC/PMU-type) — capped at Moderate by design so they don't inflate.
- Sector coverage is strongest in **waste / cross_sector / stationary_energy**, with urban-green (afolu) from MINVU and adaptation thin (mainly SUBDERE risk).
- Next: bring the **action set** in and run the match → tier per `methodology.md`; tune cutoffs here.

## C. Action financing-availability (coverage) classifier

# Financing-availability (coverage) by action — OEF, pre-Mage

**This is a COVERAGE indicator, not fundability** (see `methodology.md` §0). It asks: does a climate-relevant public funding channel of the right sector exist, is it currently usable, and can a city access it? It reflects what we have catalogued (availability bias), not the real-world probability of securing finance. Output: `financing_coverage_by_action.csv` with a `coverage_level` per action (sector-specific / broad-only / none).

In [9]:
import pandas as pd, json
inv=pd.read_csv("data/chile_finance_inventory.csv")
act=pd.read_csv("../../../../climateview/climateview-transition-elements/releases/2026-02-23/data/current_actions.csv")
inv["gpc"]=inv.gpc_sectors.apply(json.loads)
print("funds:",len(inv)," actions:",len(act))
PFX={"I":"stationary_energy","II":"transportation","III":"waste","IV":"industry","V":"afolu"}
def action_sectors(ss):
    s={PFX.get(str(ss).split(".")[0],"cross_sector")}
    if str(ss).startswith("III.4"): s.add("water")
    return s
act["sectors"]=act.subsector_number.apply(action_sectors)
print("action primary-sector spread:", act.sectors.apply(lambda x:sorted(x)[0]).value_counts().to_dict())

funds: 99  actions: 102
action primary-sector spread: {'stationary_energy': 39, 'industry': 28, 'afolu': 17, 'waste': 10, 'transportation': 8}


## Usability + city-access + per-match quality (methodology 2a-2b).

In [10]:
def usable(r):
    s=str(r.status).lower(); rec=str(r.recurrence).lower()
    return ("open" in s) or ("rolling" in s) or rec.startswith("annual") or rec.startswith("ongoing")
inv["usable_now"]=inv.apply(usable,axis=1)
def actor_ok_for_city(r):
    a=str(r.eligible_actor).lower(); ap=str(r.access_pathway).lower()
    if ("not municipal" in a) or ("no municipal" in a): return False   # e.g. "NOT municipalities"
    if "municipal" in a: return True
    if ("facilitated" in ap) or ("via municipality" in ap) or ("municipality applies" in ap): return True
    return False
inv["actor_ok"]=inv.apply(actor_ok_for_city,axis=1)
def match_quality(fund, a_sectors):
    fs=set(fund.gpc); direct=bool((fs-{"cross_sector"}) & a_sectors)
    broad=("cross_sector" in fs) or (fund.specificity=="broad")
    if not (direct or broad): return None
    if fund.detail_level=="index": return "Weak"
    if (fund.specificity=="broad") or (not direct):
        return "Moderate" if (fund.usable_now and fund.actor_ok) else "Weak"
    gaps=(0 if fund.usable_now else 1)+(0 if fund.actor_ok else 1)
    return {0:"Strong",1:"Moderate"}.get(gaps,"Weak")
print("usable funds:",int(inv.usable_now.sum()),"/",len(inv)," | city-accessible:",int(inv.actor_ok.sum()))

usable funds: 65 / 99  | city-accessible: 31


## Classify each action's coverage_level (sector-specific / broad-only / none).

In [11]:
rows=[]
for _,a in act.iterrows():
    q=[(match_quality(f,a.sectors),f.program_name,f.source_dataset) for _,f in inv.iterrows()]
    q=[x for x in q if x[0]]
    nDed=sum(v=="Strong" for v,*_ in q)      # dedicated, usable, city-accessible
    nGen=sum(v=="Moderate" for v,*_ in q)     # general/broad or sector-specific-with-gap
    nWeak=sum(v=="Weak" for v,*_ in q)
    cov="sector-specific" if nDed>=1 else ("broad-only" if nGen>=1 else "none")
    chans=[f"{n} ({d})" for v,n,d in q if v=="Strong"][:3] or [f"{n} ({d})" for v,n,d in q if v=="Moderate"][:3]
    rows.append(dict(action_id=a.action_id,action_name=a.action_name,subsector=a.subsector_number,
        sector=sorted(a.sectors)[0],coverage_level=cov,n_dedicated_channels=nDed,
        n_general_channels=nGen,example_channels="; ".join(chans)))
cov=pd.DataFrame(rows)
cov[["action_name","sector","coverage_level","n_dedicated_channels","n_general_channels"]].head(10)

,action_name,sector,coverage_level,n_dedicated_channels,n_general_channels
0,Introduce energy-efficiency standards for new ...,stationary_energy,sector-specific,2,12
1,Introduce energy-efficiency standards for new ...,stationary_energy,sector-specific,2,12
2,Support Implementation of Industrial Building ...,stationary_energy,sector-specific,2,12
3,Retrofit residential buildings for energy effi...,stationary_energy,sector-specific,2,12
4,Retrofit commercial and institutional other no...,stationary_energy,sector-specific,2,12
5,Retrofit municipal buildings for energy effici...,stationary_energy,sector-specific,2,12
6,Optimize energy efficiency and sustainability ...,stationary_energy,sector-specific,2,12
7,Adopt zero-emission bus fleets for public tran...,transportation,broad-only,0,8
8,Promote deployment of zero-emission freight fl...,transportation,broad-only,0,8
9,Electrify municipal vehicle fleets,transportation,broad-only,0,8


## Coverage summary + integrity checks (broad funds never count as dedicated).

In [12]:
print("coverage_level:"); print(cov.coverage_level.value_counts().to_string())
print(); print("sector x coverage:"); print(cov.groupby(["sector","coverage_level"]).size().unstack(fill_value=0).to_string())
assert not (cov.coverage_level=="sector-specific").all(), "all sector-specific = inflated"
assert cov.coverage_level.nunique()>1, "no spread"
print(); print("broad funds (never dedicated):", sorted(inv[inv.specificity=="broad"].program_name))
print("ASSERTIONS PASSED")

coverage_level:
coverage_level
sector-specific    66
broad-only         36

sector x coverage:
coverage_level     broad-only  sector-specific
sector                                        
afolu                       0               17
industry                   28                0
stationary_energy           0               39
transportation              8                0
waste                       0               10

broad funds (never dedicated): ['Créditos INDAP (corto y largo plazo)', 'Dirección de Arquitectura — Iniciativas de Inversión / Equipamiento social y comunitario', 'FNDR — Inversión Regional (Glosa 03 / SNI)', 'FRIL — Fondo Regional de Iniciativa Local', 'FRPD — Fondo Regional para la Productividad y el Desarrollo', 'Fondo de Infraestructura para el Desarrollo', 'Fondo de Recuperación de Ciudades (FRC)', 'Garantías / Coberturas CORFO (e.g. FOGAIN)', 'Innova Chile — I+D e innovación (clean-tech)', 'Pavimentación Participativa', 'Plan de Infraestructura en Asociación Púb

## Export

In [13]:
cov.to_csv("data/financing_coverage_by_action.csv",index=False)
print("wrote financing_coverage_by_action.csv", cov.shape)

wrote financing_coverage_by_action.csv (102, 8)


## Read-out

- **This is coverage, not fundability** (methodology §0). `sector-specific` = a dedicated, usable, city-accessible channel exists; `broad-only` = only general-purpose funds (PMU/FNDR-type) cover it; `none` = a true gap.
- Coverage is **sector-determined** here (every action in a sector shares a label) — fine for a *where-to-look* map and *gap map*, not for ranking actions.
- Transport & industry are `broad-only` because those sources are unreviewed (availability bias), **not** because such actions are inherently hard to finance.
- For *what actually gets financed in practice*, the next step is a separate **revealed** signal from award/adjudication history — deliberately not built here.